In [1]:
#!pip uninstall agentops -y

#!pip uninstall pyautogen autogen -y

#!pip install -U "autogen-agentchat==0.4.7" 
#!pip install "autogen-ext[openai]==0.4.7" chromadb
#!pip uninstall agentops opentelemetry=instrumentation opentelemetry-sdk -y

In [2]:
import asyncio
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_core.tools import FunctionTool
import chromadb

# 1. Use a NEW key - revoke the one you leaked 5 times
os.environ["OPENAI_API_KEY"] = "put api key"

# 2. OpenAI client for 0.4+ - replaces all llm_config/config_list stuff
client = OpenAIChatCompletionClient(model="gpt-4o-mini")

# 3. RAG with ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("docs")
collection.add(
    documents=["AutoGen 0.4.7 uses OpenAIChatCompletionClient. The old google.generativeai package is dead."],
    ids=["1"]
)

def search_docs(query: str) -> str:
    results = collection.query(query_texts=[query], n_results=1)
    return results['documents'][0][0] if results['documents'][0] else "No docs found"

search_tool = FunctionTool(search_docs, description="Search local docs")

# 4. Agent 1: RAG agent
rag_agent = AssistantAgent(
    name="rag_agent",
    model_client=client,
    tools=[search_tool],
    system_message="Call search_docs first. Reply TERMINATE when done."
)

# 5. Agent 2: Writer
writer_agent = AssistantAgent(
    name="writer",
    model_client=client,
    system_message="Summarize findings. Reply TERMINATE when done."
)

# 6. Run
team = RoundRobinGroupChat(
    [rag_agent, writer_agent],
    termination_condition=TextMentionTermination("TERMINATE")
)

await team.run(task="Explain the AutoGen 0.4 changes")

TaskResult(messages=[TextMessage(source='user', models_usage=None, content='Explain the AutoGen 0.4 changes', type='TextMessage'), ToolCallRequestEvent(source='rag_agent', models_usage=RequestUsage(prompt_tokens=68, completion_tokens=20), content=[FunctionCall(id='call_JKypRtXZp5QHIxEHyjV4gykq', arguments='{"query":"AutoGen 0.4 changes"}', name='search_docs')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='rag_agent', models_usage=None, content=[FunctionExecutionResult(content='AutoGen 0.4.7 uses OpenAIChatCompletionClient. The old google.generativeai package is dead.', call_id='call_JKypRtXZp5QHIxEHyjV4gykq', is_error=False)], type='ToolCallExecutionEvent'), ToolCallSummaryMessage(source='rag_agent', models_usage=None, content='AutoGen 0.4.7 uses OpenAIChatCompletionClient. The old google.generativeai package is dead.', type='ToolCallSummaryMessage'), TextMessage(source='writer', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=58), content='AutoGen 0.4.7 h